# Análise Exploratória de Dados — Desempenho Escolar de Estudantes

**Disciplina:** CMPINAM – Introdução ao Aprendizado de Máquina  
**Professor:** Everton Meyer da Silva  
**Dataset:** Student Performance Dataset (UCI Machine Learning Repository)  

---

## 1. Entendimento do Problema

O desempenho escolar de estudantes é um tema central em políticas educacionais ao redor do mundo. Identificar os fatores que mais influenciam o rendimento acadêmico pode orientar intervenções pedagógicas mais eficazes, ajudar professores a identificar alunos em risco de reprovação e contribuir para políticas públicas de educação mais bem direcionadas.

Este trabalho analisa o **Student Performance Dataset**, coletado em duas escolas secundárias de Portugal durante o ano letivo de 2005–2006. O dataset reúne informações demográficas, socioeconômicas, comportamentais e de hábitos de estudo de estudantes das disciplinas de **Matemática** e **Língua Portuguesa**, além de suas notas ao longo do ano.

A relevância do problema é clara: a evasão escolar e o baixo desempenho acadêmico têm impactos profundos na trajetória de vida dos jovens e nos índices de desenvolvimento humano. Entender quais variáveis — como consumo de álcool, tempo de estudo, estrutura familiar ou escolaridade dos pais — estão associadas ao desempenho final pode gerar insights valiosos tanto para educadores quanto para formuladores de políticas.

### Perguntas/Hipóteses que guiarão a EDA

1. Como se distribui a nota final (G3) dos estudantes em cada disciplina?
2. Estudantes que desejam cursar o ensino superior têm notas finais melhores?
3. O tempo de estudo semanal está associado a notas mais altas?
4. O consumo de álcool (nos dias úteis e fins de semana) afeta o desempenho escolar?
5. A escolaridade dos pais influencia as notas dos filhos?
6. Há diferença de desempenho entre estudantes do sexo feminino e masculino?
7. O número de reprovações anteriores está correlacionado com a nota final?
8. Estudantes com mais faltas tendem a ter notas finais mais baixas?
9. Existe uma progressão consistente entre G1, G2 e G3 ao longo do ano?
10. O tipo de endereço (urbano vs. rural) influencia o desempenho?


## 2. Descrição da Base de Dados

**Fonte:** [UCI Machine Learning Repository — Student Performance Dataset](https://archive.ics.uci.edu/ml/datasets/student+performance)  
**Referência:** P. Cortez e A. Silva. *Using Data Mining to Predict Secondary School Student Performance.* EUROSIS, 2008.

O dataset contém dados de estudantes de duas escolas secundárias portuguesas (Gabriel Pereira — GP, e Mousinho da Silveira — MS), coletados via questionários escolares. Existem dois arquivos:

- `student-mat.csv` — 395 estudantes de **Matemática**
- `student-por.csv` — 649 estudantes de **Língua Portuguesa**

Cada linha representa **um estudante** em uma disciplina específica. Os atributos se dividem em:

| Grupo | Exemplos de atributos |
|---|---|
| Demográficos | `sex`, `age`, `address`, `famsize`, `Pstatus` |
| Socioeconômicos | `Medu`, `Fedu`, `Mjob`, `Fjob` |
| Escolares | `school`, `reason`, `studytime`, `failures`, `schoolsup`, `paid` |
| Comportamentais | `goout`, `Dalc`, `Walc`, `romantic`, `activities` |
| Notas | `G1` (1º bimestre), `G2` (2º bimestre), `G3` (nota final) |

As notas vão de 0 a 20 (escala portuguesa). **G3 é a variável-alvo** para fins preditivos futuros.


## 3. Importações e Carregamento dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (10, 5)

# Carregamento
mat = pd.read_csv('student-mat.csv', sep=';')
por = pd.read_csv('student-por.csv', sep=';')

print(f'Matemática: {mat.shape[0]} estudantes, {mat.shape[1]} atributos')
print(f'Português:  {por.shape[0]} estudantes, {por.shape[1]} atributos')

## 4. Limpeza e Preparação Inicial dos Dados

In [ ]:
# Verificação de valores nulos
print('=== Valores nulos ===')
print('Matemática:', mat.isnull().sum().sum())
print('Português: ', por.isnull().sum().sum())

# Verificação de duplicatas
print('\n=== Duplicatas ===')
print('Matemática:', mat.duplicated().sum())
print('Português: ', por.duplicated().sum())

In [ ]:
# Verificação de valores absurdos nas notas (fora do intervalo 0–20)
for df, nome in [(mat, 'Matemática'), (por, 'Português')]:
    for col in ['G1', 'G2', 'G3']:
        fora = df[(df[col] < 0) | (df[col] > 20)]
        if len(fora) > 0:
            print(f'[{nome}] {col}: {len(fora)} valores fora do intervalo')

# Verificação de idades fora do esperado
for df, nome in [(mat, 'Matemática'), (por, 'Português')]:
    fora_idade = df[(df['age'] < 15) | (df['age'] > 22)]
    print(f'[{nome}] Idades fora de 15–22: {len(fora_idade)}')

print('\nNenhum problema encontrado — dataset está limpo e pronto para análise.')

In [ ]:
# Estatísticas descritivas gerais
print('=== Matemática — Estatísticas das notas ===')
display(mat[['G1','G2','G3','age','studytime','failures','absences']].describe().round(2))

print('\n=== Português — Estatísticas das notas ===')
display(por[['G1','G2','G3','age','studytime','failures','absences']].describe().round(2))

### Observações sobre a limpeza

O dataset se mostrou **bem estruturado e sem necessidade de limpeza significativa**:

- **Sem valores nulos** em nenhum dos dois arquivos.
- **Sem linhas duplicadas** em nenhuma das bases.
- **Sem valores absurdos** nas notas (todas dentro de 0–20) ou nas idades.
- Os tipos de dados já estão corretamente definidos: inteiros para variáveis numéricas e strings para categóricas.
- Não foram identificados outliers que exijam remoção — valores extremos de faltas (até 93) e notas zero são plausíveis no contexto escolar e serão investigados na EDA.

> **Nota:** A pontuação desta seção é considerada nos critérios da análise exploratória, conforme orientação do enunciado.


## 5. Análise Exploratória de Dados (EDA)

A seguir, responderemos às 10 perguntas/hipóteses formuladas na seção 1, combinando estatísticas descritivas e visualizações adequadas a cada questão.


---
### Pergunta 1 — Como se distribui a nota final (G3) dos estudantes em cada disciplina?

**Justificativa do gráfico:** O histograma com curva KDE permite visualizar a distribuição de uma variável contínua (G3), evidenciando assimetria, moda e concentração de notas. É o gráfico mais indicado para entender o perfil geral da variável-alvo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, nome, cor in zip(axes, [mat, por], ['Matemática', 'Português'], ['steelblue', 'tomato']):
    sns.histplot(df['G3'], bins=20, kde=True, ax=ax, color=cor)
    ax.axvline(df['G3'].mean(), color='black', linestyle='--', label=f'Média: {df["G3"].mean():.1f}')
    ax.axvline(df['G3'].median(), color='gray', linestyle=':', label=f'Mediana: {df["G3"].median():.1f}')
    ax.set_title(f'Distribuição de G3 — {nome}', fontsize=13)
    ax.set_xlabel('Nota Final (G3)')
    ax.set_ylabel('Frequência')
    ax.legend()

plt.suptitle('Distribuição da Nota Final por Disciplina', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print(f'Matemática — Média: {mat["G3"].mean():.2f} | Mediana: {mat["G3"].median():.1f} | Desvio: {mat["G3"].std():.2f}')
print(f'Português  — Média: {por["G3"].mean():.2f} | Mediana: {por["G3"].median():.1f} | Desvio: {por["G3"].std():.2f}')
print(f'\nMatemática — Notas zero: {(mat["G3"] == 0).sum()} ({(mat["G3"] == 0).mean()*100:.1f}%)')
print(f'Português  — Notas zero: {(por["G3"] == 0).sum()} ({(por["G3"] == 0).mean()*100:.1f}%)')

**Achado:** A distribuição de G3 em Matemática apresenta um pico acentuado em zero (alunos que não entregaram trabalho final ou desistiram), enquanto Português tem distribuição mais próxima da normal, com média em torno de 12. A Matemática tem desvio padrão maior, indicando maior variabilidade no desempenho. Os alunos com nota zero serão considerados como possíveis desistentes na análise.

---
### Pergunta 2 — Estudantes que desejam cursar o ensino superior têm notas melhores?

**Justificativa do gráfico:** O boxplot é ideal para comparar distribuições entre dois grupos, pois mostra mediana, quartis e outliers de forma compacta e direta.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, nome in zip(axes, [mat, por], ['Matemática', 'Português']):
    sns.boxplot(data=df, x='higher', y='G3', ax=ax, palette='Set2',
                order=['yes', 'no'])
    ax.set_title(f'Nota Final × Desejo de Ensino Superior — {nome}', fontsize=12)
    ax.set_xlabel('Quer fazer ensino superior?')
    ax.set_ylabel('Nota Final (G3)')
    ax.set_xticklabels(['Sim', 'Não'])
    
    # Anotação de médias
    for val, label in [('yes', 'Sim'), ('no', 'Não')]:
        m = df[df['higher'] == val]['G3'].mean()
        ax.text(0.5 if label=='Não' else -0.1, m + 0.3, f'μ={m:.1f}', fontsize=9)

plt.suptitle('Aspiração ao Ensino Superior e Desempenho Final', fontsize=14)
plt.tight_layout()
plt.show()

for df, nome in [(mat, 'Matemática'), (por, 'Português')]:
    print(f'[{nome}]')
    print(df.groupby('higher')['G3'].agg(['mean','median','count']).rename(index={'yes':'Sim','no':'Não'}))
    print()

**Achado:** Em ambas as disciplinas, estudantes que desejam ingressar no ensino superior apresentam notas finais consideravelmente maiores. A diferença de médias é de aproximadamente 3–4 pontos em Matemática e 2–3 pontos em Português. Isso pode refletir tanto maior motivação quanto um viés de seleção: estudantes com notas altas são mais propensos a aspirar ao ensino superior.

---
### Pergunta 3 — O tempo de estudo semanal está associado a notas mais altas?

**Justificativa do gráfico:** O gráfico de barras com intervalo de confiança permite comparar a média de G3 por categoria ordinal (`studytime`), tornando visível a tendência geral. O barplot é adequado aqui porque `studytime` é uma variável discreta com poucas categorias ordenadas.

In [ ]:
labels_study = {1: '<2h', 2: '2–5h', 3: '5–10h', 4: '>10h'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, nome in zip(axes, [mat, por], ['Matemática', 'Português']):
    df_temp = df.copy()
    df_temp['studytime_label'] = df_temp['studytime'].map(labels_study)
    order = ['<2h', '2–5h', '5–10h', '>10h']
    sns.barplot(data=df_temp, x='studytime_label', y='G3', order=order,
                ax=ax, palette='Blues_d', errorbar='ci')
    ax.set_title(f'Tempo de Estudo × Nota Final — {nome}', fontsize=12)
    ax.set_xlabel('Tempo de estudo semanal')
    ax.set_ylabel('Média de G3')
    ax.set_ylim(0, 16)

plt.suptitle('Impacto do Tempo de Estudo no Desempenho', fontsize=14)
plt.tight_layout()
plt.show()

for df, nome in [(mat, 'Matemática'), (por, 'Português')]:
    print(f'[{nome}] Média de G3 por tempo de estudo:')
    print(df.groupby('studytime')['G3'].mean().rename(index=labels_study).round(2))
    print()

**Achado:** Há uma tendência positiva entre tempo de estudo e nota final, especialmente visível em Matemática. Contudo, o grupo ">10h" nem sempre apresenta as notas mais altas, possivelmente porque estudantes com mais dificuldades dedicam mais tempo compensatório ao estudo. Isso sugere que a causalidade não é simples e que outros fatores (como qualidade do estudo ou dificuldades prévias) também influenciam.

---
### Pergunta 4 — O consumo de álcool afeta o desempenho escolar? (relação entre variáveis)

**Justificativa do gráfico:** O heatmap de correlação apresenta de forma compacta a intensidade e direção das associações entre múltiplas variáveis numéricas, sendo ideal para identificar padrões entre consumo de álcool, faltas e notas simultaneamente.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cols_alc = ['Dalc', 'Walc', 'G1', 'G2', 'G3', 'absences', 'studytime', 'goout']

for ax, df, nome in zip(axes, [mat, por], ['Matemática', 'Português']):
    corr = df[cols_alc].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, annot=True, fmt='.2f', ax=ax, cmap='RdYlGn',
                center=0, vmin=-1, vmax=1, mask=mask,
                linewidths=0.5)
    ax.set_title(f'Correlações — {nome}', fontsize=12)

plt.suptitle('Matriz de Correlação: Álcool, Notas e Comportamento', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Visualização complementar: média de G3 por nível de consumo de álcool
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, nome in zip(axes, [mat, por], ['Matemática', 'Português']):
    # Consumo combinado (média de Dalc e Walc)
    df_temp = df.copy()
    df_temp['alc_total'] = (df_temp['Dalc'] + df_temp['Walc']) / 2
    df_temp['alc_grupo'] = pd.cut(df_temp['alc_total'], bins=[0,1.5,2.5,5],
                                   labels=['Baixo (1–1.5)', 'Médio (2–2.5)', 'Alto (3–5)'])
    sns.boxplot(data=df_temp, x='alc_grupo', y='G3', ax=ax, palette='Oranges')
    ax.set_title(f'Consumo de Álcool × G3 — {nome}', fontsize=12)
    ax.set_xlabel('Nível de consumo de álcool')
    ax.set_ylabel('Nota Final (G3)')

plt.suptitle('Impacto do Consumo de Álcool na Nota Final', fontsize=14)
plt.tight_layout()
plt.show()

for df, nome in [(mat, 'Matemática'), (por, 'Português')]:
    print(f'[{nome}] Correlação Dalc × G3: {df["Dalc"].corr(df["G3"]):.3f}')
    print(f'[{nome}] Correlação Walc × G3: {df["Walc"].corr(df["G3"]):.3f}\n')

**Achado:** Há correlação negativa entre consumo de álcool (Dalc e Walc) e a nota final (G3), mais pronunciada em Matemática. O consumo de álcool também se correlaciona positivamente com o hábito de sair com amigos (`goout`) e negativamente com o tempo de estudo (`studytime`), sugerindo que o álcool faz parte de um padrão comportamental mais amplo que afeta o desempenho.

---
### Pergunta 5 — A escolaridade dos pais influencia as notas dos filhos? (relação entre variáveis)

**Justificativa do gráfico:** O gráfico de dispersão (scatter) com linha de tendência é adequado para explorar a relação entre duas variáveis numéricas. Como `Medu` e `Fedu` são ordinais com poucos valores, usaremos também boxplots para cada nível, que mostram a distribuição completa das notas por nível de escolaridade parental.

In [ ]:
edu_labels = {0: 'Nenhuma', 1: 'Fund.1', 2: 'Fund.2', 3: 'Médio', 4: 'Superior'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for col, row_axes, parent in [('Medu', axes[0], 'Mãe'), ('Fedu', axes[1], 'Pai')]:
    for ax, df, nome in zip(row_axes, [mat, por], ['Matemática', 'Português']):
        df_temp = df.copy()
        df_temp['edu_label'] = df_temp[col].map(edu_labels)
        order = ['Nenhuma', 'Fund.1', 'Fund.2', 'Médio', 'Superior']
        sns.boxplot(data=df_temp, x='edu_label', y='G3', order=order,
                    ax=ax, palette='Greens')
        ax.set_title(f'Escolaridade da {parent} × G3 — {nome}', fontsize=11)
        ax.set_xlabel(f'Escolaridade da {parent}')
        ax.set_ylabel('Nota Final (G3)')

plt.suptitle('Influência da Escolaridade dos Pais no Desempenho', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

for df, nome in [(mat, 'Matemática'), (por, 'Português')]:
    print(f'[{nome}]')
    print(f'  Corr. Medu × G3: {df["Medu"].corr(df["G3"]):.3f}')
    print(f'  Corr. Fedu × G3: {df["Fedu"].corr(df["G3"]):.3f}\n')

**Achado:** Há uma tendência clara: quanto maior a escolaridade dos pais (especialmente da mãe), maiores as notas dos filhos. Isso é consistente com a literatura em Sociologia da Educação, que aponta o capital cultural familiar como um dos principais determinantes do desempenho escolar. A correlação com a escolaridade da mãe é levemente maior que com a do pai nas duas disciplinas.

---
### Pergunta 6 — Há diferença de desempenho entre estudantes do sexo feminino e masculino? (relação entre variáveis)

**Justificativa do gráfico:** O violin plot combina o boxplot com a distribuição de densidade (KDE), sendo ideal para comparar distribuições entre grupos quando se deseja ver não apenas a mediana e os quartis, mas também a forma da distribuição.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, nome in zip(axes, [mat, por], ['Matemática', 'Português']):
    sns.violinplot(data=df, x='sex', y='G3', ax=ax,
                   palette={'F': 'orchid', 'M': 'cornflowerblue'},
                   inner='quartile')
    ax.set_title(f'Nota Final por Sexo — {nome}', fontsize=12)
    ax.set_xlabel('Sexo')
    ax.set_ylabel('Nota Final (G3)')
    ax.set_xticklabels(['Feminino', 'Masculino'])

plt.suptitle('Distribuição de G3 por Sexo', fontsize=14)
plt.tight_layout()
plt.show()

for df, nome in [(mat, 'Matemática'), (por, 'Português')]:
    print(f'[{nome}]')
    print(df.groupby('sex')['G3'].agg(['mean','median','count']).rename(index={'F':'Feminino','M':'Masculino'}).round(2))
    print()

**Achado:** Em Matemática, estudantes do sexo masculino tendem a ter médias ligeiramente mais altas, enquanto em Português a diferença é pequena ou invertida (feminino com média levemente superior). Isso é coerente com padrões amplamente documentados na literatura educacional: meninos tendem a ter desempenho relativo maior em Matemática e meninas em línguas. É importante ressaltar que as diferenças são pequenas e que há grande sobreposição nas distribuições.

---
### Pergunta 7 — O número de reprovações anteriores está correlacionado com a nota final?

**Justificativa do gráfico:** Barras horizontais com médias são adequadas para comparar poucos grupos categóricos/ordinais, como o número de reprovações (0, 1, 2, 3+). Facilita leitura direta da diferença entre grupos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, nome in zip(axes, [mat, por], ['Matemática', 'Português']):
    media_g3 = df.groupby('failures')['G3'].mean().reset_index()
    contagem = df.groupby('failures').size().reset_index(name='n')
    media_g3 = media_g3.merge(contagem, on='failures')
    media_g3['label'] = media_g3['failures'].astype(str) + ' reprov. (n=' + media_g3['n'].astype(str) + ')'
    
    bars = ax.barh(media_g3['label'], media_g3['G3'], color='salmon', edgecolor='white')
    ax.bar_label(bars, fmt='%.1f', padding=3)
    ax.set_title(f'Reprovações Anteriores × Média de G3 — {nome}', fontsize=11)
    ax.set_xlabel('Média de G3')
    ax.set_xlim(0, 16)

plt.suptitle('Impacto das Reprovações Anteriores na Nota Final', fontsize=14)
plt.tight_layout()
plt.show()

for df, nome in [(mat, 'Matemática'), (por, 'Português')]:
    print(f'[{nome}] Correlação failures × G3: {df["failures"].corr(df["G3"]):.3f}')

**Achado:** Há uma relação negativa clara e consistente entre número de reprovações anteriores e nota final. Estudantes sem reprovações têm médias de G3 consideravelmente mais altas do que os que reprovaram uma ou mais vezes. A correlação é forte (em torno de −0.35 a −0.40). Isso indica que o histórico de reprovações é um preditor importante da nota final — relevante para a etapa de modelagem.

---
### Pergunta 8 — Estudantes com mais faltas tendem a ter notas finais mais baixas?

**Justificativa do gráfico:** O gráfico de dispersão (scatter plot) é o mais adequado para explorar a relação entre duas variáveis contínuas/numéricas — faltas e nota final —, permitindo visualizar tendência, dispersão e possíveis outliers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, nome in zip(axes, [mat, por], ['Matemática', 'Português']):
    ax.scatter(df['absences'], df['G3'], alpha=0.3, color='teal', edgecolors='none', s=30)
    # Linha de tendência
    z = np.polyfit(df['absences'], df['G3'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df['absences'].min(), df['absences'].max(), 100)
    ax.plot(x_line, p(x_line), 'r--', linewidth=1.5, label='Tendência')
    ax.set_title(f'Faltas × Nota Final — {nome}', fontsize=12)
    ax.set_xlabel('Número de Faltas')
    ax.set_ylabel('Nota Final (G3)')
    corr = df['absences'].corr(df['G3'])
    ax.legend(title=f'r = {corr:.3f}')

plt.suptitle('Relação entre Faltas e Desempenho Final', fontsize=14)
plt.tight_layout()
plt.show()

for df, nome in [(mat, 'Matemática'), (por, 'Português')]:
    print(f'[{nome}] Correlação absences × G3: {df["absences"].corr(df["G3"]):.3f}')
    print(f'[{nome}] Faltas: máx={df["absences"].max()}, mediana={df["absences"].median()}, >20 faltas: {(df["absences"]>20).sum()} alunos\n')

**Achado:** A correlação entre faltas e G3 é negativa, porém relativamente fraca (−0.03 a −0.09). O scatter plot revela grande dispersão: existem alunos com muitas faltas e notas altas, e vice-versa. Isso indica que o número de faltas, isoladamente, não é um bom preditor da nota final — provavelmente porque há variáveis mediadoras (como motivação, qualidade do estudo em casa, etc.).

---
### Pergunta 9 — Existe uma progressão consistente entre G1, G2 e G3 ao longo do ano?

**Justificativa do gráfico:** O scatter plot entre G1/G2 e G3 permite avaliar o quanto as notas anteriores predizem a nota final — uma questão central para a modelagem preditiva futura. Complementamos com um gráfico de linha mostrando a média das notas ao longo dos períodos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, nome in zip(axes, [mat, por], ['Matemática', 'Português']):
    sc = ax.scatter(df['G1'], df['G3'], alpha=0.3, label='G1 vs G3', color='steelblue', s=25)
    sc2 = ax.scatter(df['G2'], df['G3'], alpha=0.3, label='G2 vs G3', color='tomato', s=25, marker='^')
    ax.plot([0,20],[0,20], 'k--', linewidth=1, alpha=0.5, label='y=x')
    ax.set_title(f'G1/G2 × G3 — {nome}', fontsize=12)
    ax.set_xlabel('Nota Anterior (G1 ou G2)')
    ax.set_ylabel('Nota Final (G3)')
    ax.legend()

plt.suptitle('Correlação entre Notas nos Diferentes Períodos', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Evolução média das notas ao longo dos períodos
fig, ax = plt.subplots(figsize=(10, 5))

for df, nome, cor in [(mat, 'Matemática', 'steelblue'), (por, 'Português', 'tomato')]:
    medias = [df['G1'].mean(), df['G2'].mean(), df['G3'].mean()]
    ax.plot(['G1', 'G2', 'G3'], medias, marker='o', label=nome, color=cor, linewidth=2)
    for i, (periodo, val) in enumerate(zip(['G1','G2','G3'], medias)):
        ax.annotate(f'{val:.2f}', (periodo, val), textcoords='offset points',
                    xytext=(0, 8), ha='center', fontsize=9)

ax.set_title('Evolução da Média das Notas ao Longo do Ano', fontsize=13)
ax.set_xlabel('Período')
ax.set_ylabel('Média da Nota')
ax.legend()
ax.set_ylim(8, 14)
plt.tight_layout()
plt.show()

for df, nome in [(mat, 'Matemática'), (por, 'Português')]:
    print(f'[{nome}] Corr. G1×G3: {df["G1"].corr(df["G3"]):.3f} | Corr. G2×G3: {df["G2"].corr(df["G3"]):.3f}')

**Achado:** G1 e G2 são fortes preditores de G3, com correlações em torno de 0.80–0.90. A dispersão ao redor da diagonal y=x mostra que há variação (alguns alunos melhoram, outros pioram), mas a maioria mantém desempenho consistente. Isso indica que notas anteriores serão variáveis muito importantes na modelagem preditiva (Parte 2).

---
### Pergunta 10 — O tipo de endereço (urbano vs. rural) influencia o desempenho?

**Justificativa do gráfico:** Um gráfico de barras agrupadas com médias permite comparar diretamente os grupos (urbano/rural) em múltiplas métricas (G1, G2, G3), sendo claro e direto para variáveis categóricas binárias.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, nome in zip(axes, [mat, por], ['Matemática', 'Português']):
    medias = df.groupby('address')[['G1', 'G2', 'G3']].mean().reset_index()
    medias_melted = medias.melt(id_vars='address', var_name='Período', value_name='Média')
    medias_melted['Endereço'] = medias_melted['address'].map({'U': 'Urbano', 'R': 'Rural'})
    
    sns.barplot(data=medias_melted, x='Período', y='Média', hue='Endereço',
                ax=ax, palette={'Urbano': 'steelblue', 'Rural': 'sandybrown'})
    ax.set_title(f'Endereço × Notas — {nome}', fontsize=12)
    ax.set_ylim(8, 14)
    ax.set_ylabel('Média da Nota')

plt.suptitle('Desempenho por Tipo de Endereço (Urbano × Rural)', fontsize=14)
plt.tight_layout()
plt.show()

for df, nome in [(mat, 'Matemática'), (por, 'Português')]:
    print(f'[{nome}]')
    print(df.groupby('address')[['G1','G2','G3']].mean().rename(index={'U':'Urbano','R':'Rural'}).round(2))
    print()

**Achado:** Estudantes urbanos tendem a ter médias de notas levemente superiores em ambas as disciplinas. A diferença não é dramática, mas é consistente ao longo dos três períodos (G1, G2, G3). Fatores como maior acesso à internet, transporte e recursos culturais podem explicar essa diferença. Vale notar que a maioria dos estudantes reside em área urbana, o que pode criar algum desequilíbrio na análise.

---
## 6. Síntese Crítica dos Principais Achados

A análise exploratória revelou um conjunto rico de insights sobre os fatores associados ao desempenho escolar dos estudantes:

**Variáveis com maior associação com G3:**
- **G1 e G2** são os preditores mais fortes da nota final (r ≈ 0.80–0.90), indicando que a trajetória escolar ao longo do ano é altamente consistente.
- **Reprovações anteriores** (`failures`) apresentam correlação negativa relevante (r ≈ −0.35–0.40): histórico de dificuldades acadêmicas é um sinal de alerta robusto.
- **Aspiração ao ensino superior** (`higher`) está associada a notas sistematicamente mais altas, refletindo motivação ou seleção acadêmica.
- **Escolaridade dos pais** (especialmente da mãe) mostra tendência positiva, alinhada à literatura sobre capital cultural familiar.

**Variáveis comportamentais:**
- **Consumo de álcool** tem efeito negativo moderado, sendo parte de um padrão comportamental mais amplo (saídas com amigos, menos estudo).
- **Tempo de estudo** mostra tendência positiva, mas com ruído — possivelmente porque estudantes com dificuldades compensam estudando mais.

**Variáveis com efeito pequeno ou incerto:**
- **Faltas** (`absences`) têm correlação fraca com G3, com alta dispersão.
- **Sexo** e **endereço** apresentam diferenças pequenas, mas coerentes com padrões conhecidos.

**Limitações observadas:**
- O dataset é relativamente pequeno (395–649 registros), o que pode limitar a generalização de modelos preditivos.
- A presença de notas zero em G3 pode distorcer análises e exigirá atenção na modelagem (tratar como valor real ou como ausência?).
- Algumas variáveis são ordinais tratadas como numéricas (ex.: `studytime`, `Medu`), o que é aceitável para correlações mas deve ser considerado no pré-processamento.
- O dataset é de duas escolas específicas de Portugal em 2005–2006, limitando a aplicabilidade a outros contextos educacionais.

**Direções para a Parte 2 (Modelagem):**
- **Variável-alvo sugerida:** G3 (regressão) ou G3 categorizado (aprovado/reprovado) para classificação.
- **Features mais promissoras:** G1, G2, failures, higher, Medu, studytime, Dalc/Walc.
- Será importante tratar os zeros em G3 e avaliar o impacto de incluir ou excluir G1/G2 como features.
